# 按 Eq.(34) 计算 TpMr 的 T_SR（C 发射 → D 接收）

本 notebook 按你的最新要求重整为 **C 星发射、D 星接收**，用于计算 **TpMr 的 T_SR**。

## 本版约定
- **T 星 → C 星（发射端）**
- **M 星 → D 星（接收端）**
- `d0 = (r_D(t_r) - r_C(t_r)) / |r_D(t_r) - r_C(t_r)|`，在**接收时刻** `t_r` 评估
- 即时光行时：
  \[
  \Delta t_{\rm inst} = \frac{|r_D(t_r)-r_C(t_r)|}{c_0}
  \]
- 修正后的回推时间：
  \[
  \Delta t_{\rm corr} = \Delta t_{\rm inst}\left(1+\frac{d_0\cdot v_C}{c_0}\right)
  \]
- 发射点二阶回推：
  \[
  r_e \approx r_C(t_r)-v_C(t_r)\Delta t_{\rm corr}+\frac{1}{2}a_C(t_r)\Delta t_{\rm corr}^2
  \]

## 加速度模型
不再用速度差分求加速度，而改为地球点质量引力场：

\[
a=\frac{GM}{R^2}
\]

在程序中采用对应的向量形式：

\[
\mathbf a = -\frac{\mu}{R^3}\mathbf r,\qquad \mu = GM
\]

其中 \(R=|\mathbf r|\) 为卫星到地心距离。

## 说明
- 本 notebook **保留 T_GR 外部输入项**，即仍从 `LightTime_T_GR_TpMr.xlsx` 读取 `delta_t_s`
- 若你后续想要“**不含 T_GR**”版本，我可以再给你拆出一个纯运动学版


In [19]:
import re
import numpy as np
import pandas as pd

# ----------------------------
# constants
# ----------------------------
C0 = 299792458.0                    # speed of light [m/s]
MU_EARTH = 3.986004418e14           # GM of Earth [m^3/s^2]

# ----------------------------
# input / output files
# ----------------------------
TGR_XLSX = "LightTime_T_GR_TpMr.xlsx"   # must contain columns: gps_time, delta_t_s
GNI_C = "GNI1B_2022-06-05_C_04.txt"     # C satellite (emitter)
GNI_D = "GNI1B_2022-06-05_D_04.txt"     # D satellite (receiver)

OUT_XLSX = "T_SR_Eq34_D_emit_C_recv_d0.xlsx"


In [20]:
def find_first_data_row(filepath: str) -> int:
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if re.match(r"^\s*\d+\s+", line):
                return i
    raise RuntimeError(f"No data rows found in {filepath}")


def read_gni1b(filepath: str, expected_sat: str) -> pd.DataFrame:
    cols = [
        "gps_time", "sat_id", "coord_ref",
        "x", "y", "z",
        "xerr", "yerr", "zerr",
        "vx", "vy", "vz",
        "vxerr", "vyerr", "vzerr",
        "qualflg"
    ]
    skip = find_first_data_row(filepath)
    df = pd.read_csv(filepath, sep=r"\s+", header=None, names=cols, skiprows=skip)
    df = df[(df["sat_id"] == expected_sat) & (df["coord_ref"] == "I")].copy()
    df = df[["gps_time", "x", "y", "z", "vx", "vy", "vz"]]
    df["gps_time"] = df["gps_time"].astype(np.int64)
    return df.sort_values("gps_time").reset_index(drop=True)


def read_tgr(filepath: str) -> pd.DataFrame:
    df = pd.read_excel(filepath)
    if "gps_time" not in df.columns or "delta_t_s" not in df.columns:
        raise ValueError(
            f"{filepath} must contain columns ['gps_time', 'delta_t_s'], "
            f"but got: {list(df.columns)}"
        )
    out = df[["gps_time", "delta_t_s"]].copy()
    out["gps_time"] = out["gps_time"].astype(np.int64)
    out = out.rename(columns={"delta_t_s": "T_GR_s"})
    return out.sort_values("gps_time").reset_index(drop=True)


In [21]:
def gravitational_acceleration(r_xyz: np.ndarray, mu: float = MU_EARTH) -> np.ndarray:
    """
    Earth point-mass gravity:
        a = -mu / |r|^3 * r
    whose magnitude is:
        |a| = mu / |r|^2 = GM / R^2
    """
    r_xyz = np.asarray(r_xyz, dtype=float)
    r_norm = np.linalg.norm(r_xyz)
    if r_norm == 0.0:
        raise ValueError("Position norm is zero; cannot compute gravity acceleration.")
    return -(mu / r_norm**3) * r_xyz


def solve_emission_state(
    r_T_tr: np.ndarray,   # emitter C at reception epoch t_r
    v_T_tr: np.ndarray,   # emitter C velocity at t_r
    a_T_tr: np.ndarray,   # emitter C acceleration at t_r
    r_M_tr: np.ndarray,   # receiver D at reception epoch t_r
    c: float = C0
):
    """
    Convention used here:
        T -> C satellite (emitter)
        M -> D satellite (receiver)

    Quantities evaluated at reception epoch t_r:
        d0      = (r_M - r_T) / |r_M - r_T|
        dt_inst = |r_M - r_T| / c
        dt_corr = dt_inst * (1 + (d0 · v_T)/c)

    Second-order back propagation of emission point:
        r_e ≈ r_T - v_T*dt_corr + 0.5*a_T*dt_corr^2
    """
    r_T_tr = np.asarray(r_T_tr, dtype=float)
    v_T_tr = np.asarray(v_T_tr, dtype=float)
    a_T_tr = np.asarray(a_T_tr, dtype=float)
    r_M_tr = np.asarray(r_M_tr, dtype=float)

    dr = r_M_tr - r_T_tr
    dist = float(np.linalg.norm(dr))
    if dist == 0.0:
        raise ValueError("Emitter and receiver positions coincide; cannot compute d0.")

    d0 = dr / dist
    dt_inst = dist / c
    dt_corr = dt_inst * (1.0 + float(np.dot(d0, v_T_tr)) / c)
    r_e = r_T_tr - v_T_tr * dt_corr + 0.5 * a_T_tr * (dt_corr ** 2)

    return r_e, dt_inst, dt_corr, d0


In [22]:
# ----------------------------
# read and align data
# ----------------------------
dfC = read_gni1b(GNI_C, "C")    # emitter
dfD = read_gni1b(GNI_D, "D")    # receiver
tgr = read_tgr(TGR_XLSX)

# acceleration of emitter C from point-mass Earth gravity
rC_all = dfC[["x", "y", "z"]].to_numpy(dtype=float)
aC_all = np.apply_along_axis(gravitational_acceleration, 1, rC_all)

dfC_acc = dfC.copy()
dfC_acc["ax"] = aC_all[:, 0]
dfC_acc["ay"] = aC_all[:, 1]
dfC_acc["az"] = aC_all[:, 2]
dfC_acc["a_mag"] = np.linalg.norm(aC_all, axis=1)

# align by gps_time
df = (
    dfC_acc.merge(dfD, on="gps_time", suffixes=("_C", "_D"))
           .merge(tgr, on="gps_time", how="inner")
           .sort_values("gps_time")
           .reset_index(drop=True)
)

print("rows:", len(df))
df.head()


rows: 86400


,gps_time,x_C,y_C,z_C,vx_C,vy_C,vz_C,ax,ay,az,a_mag,x_D,y_D,z_D,vx_D,vy_D,vz_D,T_GR_s
0,707659200,3.716399e+06,3.208044e+06,4.780815e+06,-4149.229192,-3352.914250,5461.061418,-4.603380,-3.973698,-5.921836,8.488199,3.820699e+06,3.292583e+06,4.639412e+06,-4030.053419,-3250.486988,5610.213439,8.420871e-13
1,707659201,3.712248e+06,3.204689e+06,4.786273e+06,-4153.820680,-3356.877897,5455.131498,-4.598258,-3.969559,-5.928623,8.488224,3.816666e+06,3.289331e+06,4.645020e+06,-4034.774291,-3254.555538,5604.458237,8.420898e-13
2,707659202,3.708092e+06,3.201331e+06,4.791725e+06,-4158.407018,-3360.837382,5449.194835,-4.593130,-3.965416,-5.935402,8.488248,3.812629e+06,3.286074e+06,4.650621e+06,-4039.490160,-3258.620051,5598.696106,8.420925e-13
3,707659203,3.703931e+06,3.197968e+06,4.797171e+06,-4162.988200,-3364.792697,5443.251438,-4.587996,-3.961268,-5.942174,8.488273,3.808587e+06,3.282814e+06,4.656217e+06,-4044.201020,-3262.680523,5592.927053,8.420952e-13
4,707659204,3.699766e+06,3.194601e+06,4.802612e+06,-4167.564221,-3368.743840,5437.301313,-4.582856,-3.957114,-5.948938,8.488297,3.804541e+06,3.279549e+06,4.661807e+06,-4048.906867,-3266.736949,5587.151084,8.420979e-13


In [23]:
# ----------------------------
# Eq.(34) calculation of T_SR
# ----------------------------
rows = []

for row in df.itertuples(index=False):
    # C = emitter (T), D = receiver (M)
    rC_tr = np.array([row.x_C, row.y_C, row.z_C], dtype=float)
    vC_tr = np.array([row.vx_C, row.vy_C, row.vz_C], dtype=float)
    aC_tr = np.array([row.ax, row.ay, row.az], dtype=float)

    rD_tr = np.array([row.x_D, row.y_D, row.z_D], dtype=float)

    r_e, dt_inst, dt_corr, d0 = solve_emission_state(
        r_T_tr=rC_tr,
        v_T_tr=vC_tr,
        a_T_tr=aC_tr,
        r_M_tr=rD_tr,
        c=C0
    )

    # Eq.(34) kinematic quantities all use emitter C
    d0_v = float(np.dot(d0, vC_tr))
    d0_a = float(np.dot(d0, aC_tr))
    v2 = float(np.dot(vC_tr, vC_tr))
    va = float(np.dot(vC_tr, aC_tr))

    T_GR = float(row.T_GR_s)

    # Eq.(34) terms (Delta t_media neglected)
    term1 = dt_inst * d0_v / C0
    term2 = -(dt_inst ** 2) * d0_a / (2.0 * C0)
    term3_num = (dt_inst ** 2) * (-(d0_a * d0_v) - 0.5 * va) + (dt_inst / 2.0) * ((d0_v ** 2) + v2)
    term3 = term3_num / (C0 ** 2)
    term4 = (dt_inst * d0_v * v2) / (C0 ** 3)
    term5 = (T_GR * d0_v) / C0

    T_SR = term1 + term2 + term3 + term4 + term5
    rho_SR = C0 * T_SR

    rows.append(
        (
            int(row.gps_time),
            T_SR, rho_SR,
            dt_inst, dt_corr, T_GR,
            r_e[0], r_e[1], r_e[2],
            d0[0], d0[1], d0[2],
            d0_v, d0_a, v2, va,
            row.ax, row.ay, row.az, row.a_mag
        )
    )

out = pd.DataFrame(
    rows,
    columns=[
        "gps_time",
        "T_SR_s", "rho_SR_m",
        "dt_inst_s", "dt_corr_s", "T_GR_s",
        "re_x", "re_y", "re_z",
        "d0_x", "d0_y", "d0_z",
        "d0_dot_vC_mps", "d0_dot_aC_mps2",
        "vC2_m2ps2", "vC_dot_aC_m2ps3",
        "aC_x_mps2", "aC_y_mps2", "aC_z_mps2", "aC_mag_mps2"
    ]
)

out.to_excel(OUT_XLSX, index=False)
out.head()


,gps_time,T_SR_s,rho_SR_m,dt_inst_s,dt_corr_s,T_GR_s,re_x,re_y,re_z,d0_x,d0_y,d0_z,d0_dot_vC_mps,d0_dot_aC_mps2,vC2_m2ps2,vC_dot_aC_m2ps3,aC_x_mps2,aC_y_mps2,aC_z_mps2,aC_mag_mps2
0,707659200,-1.656052e-08,-4.964720,0.00065,0.00065,8.420871e-13,3.716402e+06,3.208046e+06,4.780811e+06,0.534905,0.433562,-0.725190,-7633.447095,0.109245,5.828133e+07,84.434666,-4.603380,-3.973698,-5.921836,8.488199
1,707659201,-1.656052e-08,-4.964720,0.00065,0.00065,8.420898e-13,3.712251e+06,3.204692e+06,4.786270e+06,0.535515,0.434087,-0.724425,-7633.446221,0.109270,5.828132e+07,84.246639,-4.598258,-3.969559,-5.928623,8.488224
2,707659202,-1.656052e-08,-4.964719,0.00065,0.00065,8.420925e-13,3.708094e+06,3.201333e+06,4.791722e+06,0.536126,0.434612,-0.723659,-7633.445323,0.109295,5.828130e+07,84.058399,-4.593130,-3.965416,-5.935402,8.488248
3,707659203,-1.656052e-08,-4.964718,0.00065,0.00065,8.420952e-13,3.703934e+06,3.197970e+06,4.797168e+06,0.536735,0.435136,-0.722892,-7633.444399,0.109320,5.828129e+07,83.869947,-4.587996,-3.961268,-5.942174,8.488273
4,707659204,-1.656051e-08,-4.964717,0.00065,0.00065,8.420979e-13,3.699768e+06,3.194603e+06,4.802608e+06,0.537344,0.435659,-0.722124,-7633.443450,0.109345,5.828127e+07,83.681283,-4.582856,-3.957114,-5.948938,8.488297


In [24]:
print("rows:", len(out))
print("T_SR_s min / mean / max:", out["T_SR_s"].min(), out["T_SR_s"].mean(), out["T_SR_s"].max())
print("rho_SR_m min / mean / max:", out["rho_SR_m"].min(), out["rho_SR_m"].mean(), out["rho_SR_m"].max())
print("dt_inst_s min / mean / max:", out["dt_inst_s"].min(), out["dt_inst_s"].mean(), out["dt_inst_s"].max())
print("dt_corr_s min / mean / max:", out["dt_corr_s"].min(), out["dt_corr_s"].mean(), out["dt_corr_s"].max())
print("aC_mag_mps2 min / mean / max:", out["aC_mag_mps2"].min(), out["aC_mag_mps2"].mean(), out["aC_mag_mps2"].max())
print("saved to:", OUT_XLSX)


rows: 86400
T_SR_s min / mean / max: -1.6562420863611805e-08 -1.6474316735075e-08 -1.6369926773824244e-08
rho_SR_m min / mean / max: -4.965288861132666 -4.938875907878736 -4.90758058480478
dt_inst_s min / mean / max: 0.0006466246386249014 0.000648716880142773 0.0006504684720892359
dt_corr_s min / mean / max: 0.0006466082682809575 0.0006487004054077185 0.0006504519092753424
aC_mag_mps2 min / mean / max: 8.39834155066603 8.444206123761015 8.493096231623163
saved to: T_SR_Eq34_D_emit_C_recv_d0.xlsx


## 输出字段说明

- `T_SR_s`：按 Eq.(34) 计算得到的 T_SR
- `rho_SR_m`：对应距离量，`rho_SR = c0 * T_SR`
- `dt_inst_s`：即时几何光行时
- `dt_corr_s`：带一阶速度修正后的回推时间
- `re_x, re_y, re_z`：二阶回推得到的发射点位置
- `d0_x, d0_y, d0_z`：接收时刻评估的视线单位向量
- `d0_dot_vC_mps`：\(d_0 \cdot v_C\)
- `d0_dot_aC_mps2`：\(d_0 \cdot a_C\)
- `aC_x_mps2, aC_y_mps2, aC_z_mps2`：由地球点质量模型计算的 C 星加速度分量
- `aC_mag_mps2`：加速度模长，满足 \(|a_C| = GM/R^2\)
